# Pav-Bhaji Text Classification

**Objective:** classify an Instagram post as Pav Bhaji (`1`) or Not Pav Bhaji (`0`) using **text metadata only**.

> No CNN, image pixels, image embeddings, or computer-vision models are used.

The image folders provide the ground-truth labels and are used only to link each image filename to the corresponding JSON metadata record.

In [ ]:
from pathlib import Path
import json, os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

DATA_DIR = Path("dataset")   # change if necessary
RANDOM_STATE = 42

## 1. Inspect the JSON structure

In [ ]:
with open(DATA_DIR / "pavbhaji.json", "r", encoding="utf-8") as f:
    records = json.load(f)

print("Number of JSON records:", len(records))
print("Top-level fields:")
print(sorted(records[0].keys()))
print("\nExample record keys/types:")
for k, v in records[0].items():
    print(f"{k}: {type(v).__name__}")

## 2. Match image labels to JSON metadata

In [ ]:
def url_basename(url):
    if not url:
        return None
    return os.path.basename(str(url).split("?")[0])

label_by_filename = {}
for label in (0, 1):
    for p in (DATA_DIR / "images" / str(label)).iterdir():
        if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}:
            label_by_filename[p.name] = label

rows = []
for idx, record in enumerate(records):
    candidates = [url_basename(record.get("display_url")), url_basename(record.get("thumbnail_src"))]
    image_name = next((x for x in candidates if x in label_by_filename), None)
    if image_name is None:
        continue

    try:
        caption = record["edge_media_to_caption"]["edges"][0]["node"]["text"] or ""
    except (KeyError, IndexError, TypeError):
        caption = ""

    tags = " ".join(record.get("tags") or [])
    rows.append({
        "json_index": idx,
        "image": image_name,
        "label": label_by_filename[image_name],
        "caption": caption,
        "tags": tags,
        "likes": (record.get("edge_media_preview_like") or {}).get("count"),
        "comments": (record.get("edge_media_to_comment") or {}).get("count"),
        "is_video": record.get("is_video"),
        "location": record.get("location", {}).get("name", "") if isinstance(record.get("location"), dict) else "",
        "timestamp": record.get("taken_at_timestamp")
    })

df = pd.DataFrame(rows)
df["text_raw"] = (df["caption"].fillna("") + " " + df["tags"].fillna("")).str.strip()
df.shape

In [ ]:
print("Matched rows:", len(df))
print("\nClass distribution:")
print(df["label"].value_counts().sort_index())
print("\nMissing values:")
print(df.isna().sum())
print("\nDuplicate combined-text rows:", df["text_raw"].duplicated().sum())

## 3. Feature relevance

In [ ]:
feature_relevance = pd.DataFrame([
    ["caption / description", "Text", "High", "Contains semantic descriptions of food and posts."],
    ["tags / hashtags", "Text", "High", "Food-specific hashtags are useful lexical signals."],
    ["location.name", "Categorical/text", "Low-Medium", "May provide contextual information, but is not a direct food signal."],
    ["likes / comments", "Numeric", "Low", "Engagement is not a reliable semantic indicator of the food."],
    ["is_video", "Boolean", "Low", "Media type does not directly identify Pav Bhaji."],
    ["timestamp", "Numeric/date", "Low", "Time is not a direct food-content feature."],
    ["post id / shortcode", "Identifier", "None", "Unique identifiers should not be predictive features."],
    ["image filename", "Identifier", "None", "Used only to obtain the ground-truth label and link metadata."],
], columns=["Feature", "Type", "Relevance", "Reason"])
feature_relevance

## 4. Text preprocessing and EDA

In [ ]:
def clean_text(text):
    text = "" if pd.isna(text) else str(text)
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"[^\w#\s]", " ", text)
    text = re.sub(r"_+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

df["text"] = df["text_raw"].map(clean_text)

print("Average words:", df["text"].str.split().str.len().mean())
print("Median words:", df["text"].str.split().str.len().median())
print("Contains 'pavbhaji':", df["text"].str.contains("pavbhaji", regex=False).sum(), "/", len(df))
print("Contains 'pav bhaji':", df["text"].str.contains("pav bhaji", regex=False).sum(), "/", len(df))

In [ ]:
counts = df["label"].value_counts().sort_index()
plt.figure(figsize=(6,4))
plt.bar(["0: Non-Pav Bhaji", "1: Pav Bhaji"], [counts[0], counts[1]])
plt.ylabel("Number of samples")
plt.title("Class Distribution")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7,4))
plt.hist(df["text"].str.split().str.len(), bins=20)
plt.xlabel("Word count")
plt.ylabel("Number of posts")
plt.title("Text Length Distribution")
plt.tight_layout()
plt.show()

### Important dataset observation

The dataset contains 452 image/metadata matches: 269 non-Pav-Bhaji and 183 Pav-Bhaji. The JSON contains 1,500 metadata records, but only the records that can be matched to one of the supplied labeled images are used for supervised learning.

Also, the data was supplied as posts containing `#pavbhaji` in their description/comment. In the matched subset, the text contains `pavbhaji` in 443 of 452 rows. Therefore, the task is **not** simply detecting the word "pavbhaji"; the classifier must learn other textual patterns associated with the image-derived labels. This also creates a dataset-selection bias that should be mentioned as a limitation.

## 5. TF-IDF + classification

In [ ]:
X = df["text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

models = {
    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1,2), sublinear_tf=True)),
        ("classifier", LogisticRegression(max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE))
    ]),
    "Linear SVM": Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1,2), sublinear_tf=True)),
        ("classifier", LinearSVC(C=0.3, class_weight="balanced", random_state=RANDOM_STATE))
    ]),
    "Multinomial Naive Bayes": Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1,2))),
        ("classifier", MultinomialNB())
    ])
}

holdout_results = []
fitted_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    holdout_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0)
    })
    fitted_models[name] = model

holdout_results = pd.DataFrame(holdout_results)
holdout_results

## 6. Five-fold stratified cross-validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []

for name, model in models.items():
    scores = cross_validate(
        model, X, y, cv=cv,
        scoring=["accuracy", "precision", "recall", "f1"]
    )
    cv_rows.append({
        "Model": name,
        "CV Accuracy Mean": scores["test_accuracy"].mean(),
        "CV Precision Mean": scores["test_precision"].mean(),
        "CV Recall Mean": scores["test_recall"].mean(),
        "CV F1 Mean": scores["test_f1"].mean()
    })

cv_results = pd.DataFrame(cv_rows)
cv_results

## 7. Confusion matrix and final model

In [ ]:
final_model_name = "Logistic Regression"
final_model = fitted_models[final_model_name]
final_pred = final_model.predict(X_test)

print(classification_report(
    y_test, final_pred,
    target_names=["Non-Pav Bhaji", "Pav Bhaji"],
    zero_division=0
))

cm = confusion_matrix(y_test, final_pred)
ConfusionMatrixDisplay(cm, display_labels=["Non-Pav Bhaji", "Pav Bhaji"]).plot(values_format="d")
plt.title("Logistic Regression Confusion Matrix")
plt.tight_layout()
plt.show()

## 8. Example prediction

The model predicts the **class of the metadata associated with an image**, not the pixels of the image itself.

`1 = Pav Bhaji`  
`0 = Not Pav Bhaji`

In [ ]:
final_model.fit(X, y)

examples = [
    "delicious pav bhaji with butter pav",
    "chicken tikka with mint chutney",
    "street food and masala bhaji"
]
predictions = final_model.predict(examples)

pd.DataFrame({
    "Text": examples,
    "Prediction": predictions,
    "Label": np.where(predictions == 1, "Pav Bhaji", "Not Pav Bhaji")
})

## 9. Conclusion

This solution uses:

**Instagram metadata → text preprocessing → TF-IDF unigram/bigram features → classical ML classifier**

The supplied images are used only for the ground-truth label and metadata matching. The main limitation is that the dataset is preselected around the `#pavbhaji` topic and the supplied matched subset is relatively small (452 rows), so performance should not be interpreted as general image-recognition accuracy.